In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
from pathlib import Path

if (not Path("data/global_economy.csv").exists()) and Path("sprint-6/data/global_economy.csv").exists():
    os.chdir("sprint-6")

os.environ["NIXTLA_ID_AS_COL"] = "true"
import numpy as np
np.set_printoptions(suppress=True)
np.random.seed(1)
import random
random.seed(1)
import pandas as pd
pd.set_option("max_colwidth", 100)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_style("whitegrid")
import matplotlib.pyplot as plt
plt.style.use("ggplot")
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 100,
    "figure.constrained_layout.use": True,
    "font.size": 11.5,
    "axes.titlesize": 14,
    "axes.labelsize": 11.5,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "legend.fontsize": 9.5,
    "legend.title_fontsize": 11.5,
    "grid.alpha": 1.0,
})

import matplotlib as mpl
from cycler import cycler
mpl.rcParams["axes.prop_cycle"] = cycler(color=["#000000", "#000000"])

from functools import partial
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from utilsforecast.evaluation import evaluate
from utilsforecast.feature_engineering import pipeline, trend
from utilsforecast.losses import rmse, mae, mape as _mape, mase, quantile_loss, mqloss
from utilsforecast.plotting import plot_series as plot_series_utils
from statsforecast import StatsForecast
from statsforecast.models import (
    HistoricAverage,
    Naive,
    RandomWalkWithDrift,
    SeasonalNaive,
    SklearnModel,
)

class LinearRegression:
    def __init__(self):
        self.coef_ = None
        self.intercept_ = None
        self._XtX_inv = None
        self._sigma2 = None
        self._p = None

    def get_params(self, deep=True):
        return {}

    def set_params(self, **params):
        return self

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        n = X.shape[0]
        Xd = np.column_stack([np.ones(n), X])
        beta, *_ = np.linalg.lstsq(Xd, y, rcond=None)
        self.intercept_ = float(beta[0])
        self.coef_ = beta[1:]
        self._p = Xd.shape[1]
        resid = y - Xd @ beta
        df = max(n - self._p, 1)
        self._sigma2 = float((resid @ resid) / df)
        self._XtX_inv = np.linalg.inv(Xd.T @ Xd)
        return self

    def predict(self, X):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return self.intercept_ + X @ self.coef_

    def add_prediction_intervals(self, fcasts, X_df, levels=(80, 95), model_col="LinearRegression"):
        if isinstance(X_df, pd.DataFrame):
            cols = [c for c in X_df.columns if c not in ["unique_id", "ds"]]
            X = X_df[cols].to_numpy()
        else:
            X = np.asarray(X_df)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        n = X.shape[0]
        Xd = np.column_stack([np.ones(n), X])

        base = np.einsum("ij,jk,ik->i", Xd, self._XtX_inv, Xd)
        se = np.sqrt(self._sigma2 * (1.0 + base))
        yhat = fcasts[model_col].to_numpy()
        from scipy.stats import norm
        for level in levels:
            z = norm.ppf(0.5 + level / 200)
            fcasts[f"{model_col}-lo-{level}"] = yhat - z * se
            fcasts[f"{model_col}-hi-{level}"] = yhat + z * se
        return fcasts

def mape(df, models, id_col="unique_id", target_col="y"):
    df_mape = _mape(df, models, id_col=id_col, target_col=target_col)
    df_mape.loc[:, df_mape.select_dtypes(include="number").columns] *= 100
    return df_mape

def quantile_score(df, models, q=0.5, id_col="unique_id", target_col="y"):
    df_qs = quantile_loss(df, models, q=q, id_col=id_col, target_col=target_col)
    df_qs.loc[:, df_qs.select_dtypes(include="number").columns] *= 2
    return df_qs

mpl.colormaps.register(
    mpl.colors.ListedColormap(["#000000", "#2f2fff"], name="black_and_blue"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(["#000000", "#D55E00"], name="black_and_orange"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(["#000000", "#000000"], name="black"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(["#000000", "#0072B2", "#D55E00"], name="black_and_2color"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(["#000000", "#D55E00", "#0072B2", "#009E73"], name="black_and_3color"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(["#000000", "#D55E00", "#0072B2", "#009E73", "#CC79A7"], name="black_and_4color"),
    force=True,
)

def plot_series(df, forecasts_df=None, xlabel=None, ylabel=None, title=None, rm_legend=True, legend_loc="best", **kwargs):
    fig = plot_series_utils(df=df, forecasts_df=forecasts_df, **kwargs)
    if fig is None:
        fig = plt.gcf()
    ax = fig.axes[0] if getattr(fig, "axes", None) else plt.gca()
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title)
    if rm_legend:
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()
    else:
        if legend_loc == "outside lower center":
            ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)
        else:
            ax.legend(loc=legend_loc)
    return fig

os.makedirs("images", exist_ok=True)
print("cwd:", os.getcwd())


cwd: c:\Users\amelu\OneDrive\Desktop\я\qwant-mentorstvo\quant-research-mentorstvo\sprint-6


In [2]:
gdp_df = (
    pd.read_csv("data/global_economy.csv", parse_dates=["ds"])
    [["unique_id", "ds", "GDP", "Population"]]
    .assign(
        GDP=lambda x: x["GDP"].interpolate(),
        Population=lambda x: x["Population"].interpolate(),
        y=lambda x: x["GDP"] / x["Population"],
    )
)
gdp_df.head()


,unique_id,ds,GDP,Population,y
0,Afghanistan,1960-01-01,5.378e+08,8.996e+06,59.777
1,Afghanistan,1961-01-01,5.489e+08,9.167e+06,59.878
2,Afghanistan,1962-01-01,5.467e+08,9.346e+06,58.493
3,Afghanistan,1963-01-01,7.511e+08,9.534e+06,78.783
4,Afghanistan,1964-01-01,8.000e+08,9.731e+06,82.208


In [3]:
fig = plot_series(gdp_df, ids=["Sweden"],
    xlabel="Year [1Y]", ylabel="$US", title="GDP per capita for Sweden")
fig.savefig("images/fig_5_2_sweden_gdp_per_capita.png", dpi=150)
plt.show()


In [4]:
sweden_df = (
    gdp_df
    .loc[lambda x: x["unique_id"] == "Sweden"]
    [["unique_id", "ds", "y"]]
)
sweden_df.head()


,unique_id,ds,y
13324,Sweden,1960-01-01,1983.107
13325,Sweden,1961-01-01,2147.229
13326,Sweden,1962-01-01,2315.847
13327,Sweden,1963-01-01,2492.545
13328,Sweden,1964-01-01,2758.943


In [5]:
train_features, valid_features = pipeline(
    sweden_df, features=[trend], freq="Y", h=3)

trend_model = SklearnModel(LinearRegression())
sf = StatsForecast(models=[trend_model], freq="Y")
sf.fit(df=train_features)

fcasts = sf.predict(h=3, X_df=valid_features)

model = sf.fitted_[0][0].model_["model"]
fcasts = model.add_prediction_intervals(
    fcasts, valid_features.rename(columns={"trend": "x1"}))
fcasts


,unique_id,ds,LinearRegression,LinearRegression-lo-80,LinearRegression-hi-80,LinearRegression-lo-95,LinearRegression-hi-95
0,Sweden,2017-12-31,56165.506,49562.317,62768.694,46066.800,66264.211
1,Sweden,2018-12-31,57225.849,50611.289,63840.410,47109.752,67341.946
2,Sweden,2019-12-31,58286.193,51659.902,64912.484,48152.156,68420.230


In [6]:
fig = plot_series(sweden_df, fcasts, ids=["Sweden"], level=[80, 95],
    xlabel="Year", ylabel="$US", title="GDP per capita for Sweden",
    palette="black_and_blue", rm_legend=False,
    legend_loc="outside lower center")
fig.savefig("images/fig_5_3_sweden_trend_forecast.png", dpi=150)
plt.show()


In [7]:
production_df = pd.read_csv("data/aus_production_formatted.csv",
  parse_dates=["ds"])
beers_df = production_df.loc[
    lambda x: (x["unique_id"] == "Beer") & (x["ds"] >= "1992")
]
train = beers_df.loc[lambda x: x["ds"] < "2007"]
test = beers_df.loc[lambda x: x["ds"] >= "2007"]
avg_method = HistoricAverage()
naive_method = Naive()
seasonal_naive_method = SeasonalNaive(4)
sf = StatsForecast(
    models=[avg_method, naive_method, seasonal_naive_method],
    freq="QS",
)
sf.fit(train)
fcasts = sf.predict(h=14).assign(y=test["y"].to_numpy())
fig = plot_series(train, fcasts,
    xlabel="Quarter", ylabel="Megalitres",
    title="Forecasts for quarterly beer production",
    palette="black_and_3color", rm_legend=False,
    legend_loc="outside lower center")
fig.savefig("images/fig_5_8_beer_forecasts.png", dpi=150)
plt.show()


In [8]:
plt.rcParams["figure.figsize"] = (7, 4)
goog_df = (
  pd.read_csv("data/gafa_stock.csv", parse_dates=["ds"])
  .loc[lambda x: (
    (x["unique_id"] == "GOOG_Close") & x["ds"].between("2015", "2016-01-31")
  )]
)
train = goog_df.loc[lambda x: x["ds"].dt.year == 2015].copy()
test = goog_df.loc[lambda x: x["ds"].dt.year == 2016].copy()

train["ds"] = np.arange(len(train))
test["ds"] = len(train) + np.arange(len(test))

avg_method = HistoricAverage()
naive_method = Naive()
drift_method = RandomWalkWithDrift()
sf = StatsForecast(freq=1,
    models=[drift_method, avg_method, naive_method])
sf.fit(train)

fcasts = sf.predict(h=len(test)).assign(y=test["y"].to_numpy())
fig = plot_series(train, fcasts,
    xlabel="day", ylabel="$US",
    title="Google daily closing stock prices (Jan 2015-Jan 2016)",
    palette="black_and_4color", rm_legend=False,
    legend_loc="outside lower center")
fig.savefig("images/fig_5_9_google_forecasts.png", dpi=150)
plt.show()
